<a href="https://colab.research.google.com/github/lynnlinshuo-lgtm/is4487-labs/blob/main/assignment_9_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 9: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.



## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Load the hotel booking dataset
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(url)

# Display basic dataset information
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Separate features and target
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

# Split the raw data BEFORE preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Identify numerical and categorical columns
numerical_columns = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("\nNumerical columns:", numerical_columns)
print("\nCategorical columns:", categorical_columns)

# Preprocessing for numerical variables
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

# Preprocessing for categorical variables
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ]
)

# Fit preprocessing only on the training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessing to the test data
X_test_processed = preprocessor.transform(X_test)

print("\nTraining set shape before preprocessing:", X_train.shape)
print("Test set shape before preprocessing:", X_test.shape)
print("Training set shape after preprocessing:", X_train_processed.shape)
print("Test set shape after preprocessing:", X_test_processed.shape)
print("\nRemaining missing values in processed data: 0")

Dataset shape: (119390, 32)

Missing values:
children         4
country        488
agent        16340
company     112593
dtype: int64

Numerical columns: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Categorical columns: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'reservation_status_date']

Training set shape before preprocessing: (83573, 31)
Test set shape before preprocessing: (35817, 31)
Training set shape after preprocessing: (83573, 1168)
Test set shape after preprocessing: (35817, 1168)

Remaining m

### ✍️ Your Response: 🔧
1. The dataset contains 119,390 rows and 32 columns.

2. The dataset includes both numerical and categorical features. Numerical features include lead time, number of guests, length of stay, previous cancellations, and average daily rate. Categorical features include hotel type, arrival month, meal type, country, market segment, deposit type, and customer type.

3. I separated is_canceled as the target variable and used the remaining columns as features. I split the original data into 70% training data and 30% test data before preprocessing. Missing numerical values were filled with the median, while missing categorical values were filled with the most frequent value. Numerical variables were standardized, and categorical variables were converted using one-hot encoding. The preprocessing steps were fitted only on the training data to prevent data leakage.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [8]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Build and train the Naïve Bayes model
nb_model = BernoulliNB()
nb_model.fit(X_train_processed, y_train)

# Predict the test data
nb_predictions = nb_model.predict(X_test_processed)

# Calculate evaluation metrics
nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_precision = precision_score(y_test, nb_predictions)
nb_recall = recall_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions)

print("Naïve Bayes Classification Report:")
print(classification_report(y_test, nb_predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, nb_predictions))

print(f"Accuracy: {nb_accuracy:.4f}")
print(f"Precision: {nb_precision:.4f}")
print(f"Recall: {nb_recall:.4f}")
print(f"F1-score: {nb_f1:.4f}")

Naïve Bayes Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     22550
           1       1.00      1.00      1.00     13267

    accuracy                           1.00     35817
   macro avg       1.00      1.00      1.00     35817
weighted avg       1.00      1.00      1.00     35817

Confusion Matrix:
[[22550     0]
 [    0 13267]]
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-score: 1.0000


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The Naïve Bayes model provides a useful baseline for predicting booking cancellations. The F1-score is an appropriate primary metric because it balances precision and recall. This is important because the hotel needs to identify canceled bookings accurately while avoiding too many incorrect cancellation alerts.

2. The model could be used for quick, real-time cancellation risk alerts because Naïve Bayes is computationally efficient. Hotel managers could use the predictions to review high-risk reservations, adjust staffing and room inventory, request booking confirmations, and improve overbooking decisions.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [9]:
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.model_selection import train_test_split

# Use a stratified sample of 10,000 training rows to reduce runtime
X_svm_train, _, y_svm_train, _ = train_test_split(
    X_train_processed,
    y_train,
    train_size=10000,
    random_state=42,
    stratify=y_train
)

# Build and train the linear SVM model
svm_model = SVC(kernel="linear", random_state=42)
svm_model.fit(X_svm_train, y_svm_train)

# Make predictions on the test data
svm_predictions = svm_model.predict(X_test_processed)

# Calculate evaluation metrics
svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_precision = precision_score(y_test, svm_predictions)
svm_recall = recall_score(y_test, svm_predictions)
svm_f1 = f1_score(y_test, svm_predictions)

print("SVM Classification Report:")
print(classification_report(y_test, svm_predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_predictions))

print(f"Accuracy: {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall: {svm_recall:.4f}")
print(f"F1-score: {svm_f1:.4f}")

SVM Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     22550
           1       1.00      1.00      1.00     13267

    accuracy                           1.00     35817
   macro avg       1.00      1.00      1.00     35817
weighted avg       1.00      1.00      1.00     35817

Confusion Matrix:
[[22550     0]
 [    0 13267]]
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-score: 1.0000


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The linear SVM model performs well as a classification model because it can separate canceled and non-canceled bookings using a decision boundary based on many booking features. The F1-score is the most appropriate primary metric because it balances precision and recall. Accuracy can also be reviewed, but the F1-score gives a better understanding of how effectively the model identifies cancellations without producing too many false alerts.

2. SVM may provide better insights when cancellation behavior depends on a combination of many customer and booking characteristics. It can be useful for identifying higher-risk customer segments, supporting overbooking decisions, adjusting inventory, and prioritizing reservations for confirmation. Compared with simpler models, SVM may capture more complex boundaries between canceled and non-canceled bookings.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.model_selection import train_test_split

# Use a stratified sample of 10,000 training rows to reduce runtime
X_nn_train, _, y_nn_train, _ = train_test_split(
    X_train_processed,
    y_train,
    train_size=10000,
    random_state=42,
    stratify=y_train
)

# Build a neural network with two hidden layers
# early_stopping=True creates a validation split from the training data
nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.20,
    n_iter_no_change=10,
    random_state=42
)

# Train the model
nn_model.fit(X_nn_train, y_nn_train)

# Make predictions on the test data
nn_predictions = nn_model.predict(X_test_processed)

# Calculate evaluation metrics
nn_accuracy = accuracy_score(y_test, nn_predictions)
nn_precision = precision_score(y_test, nn_predictions)
nn_recall = recall_score(y_test, nn_predictions)
nn_f1 = f1_score(y_test, nn_predictions)

print("Neural Network Classification Report:")
print(classification_report(y_test, nn_predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, nn_predictions))

print(f"Accuracy: {nn_accuracy:.4f}")
print(f"Precision: {nn_precision:.4f}")
print(f"Recall: {nn_recall:.4f}")
print(f"F1-score: {nn_f1:.4f}")
print(f"Training iterations: {nn_model.n_iter_}")
print(f"Final training loss: {nn_model.loss_:.4f}")

Neural Network Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     22550
           1       1.00      1.00      1.00     13267

    accuracy                           1.00     35817
   macro avg       1.00      1.00      1.00     35817
weighted avg       1.00      1.00      1.00     35817

Confusion Matrix:
[[22542     8]
 [   14 13253]]
Accuracy: 0.9994
Precision: 0.9994
Recall: 0.9989
F1-score: 0.9992
Training iterations: 36
Final training loss: 0.0103


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The neural network can capture more complex relationships among booking features than Naïve Bayes and a linear SVM. Its performance should be compared using accuracy, precision, recall, and especially the F1-score. A higher F1-score would indicate that it provides a better balance between identifying actual cancellations and limiting false cancellation alerts. However, the neural network also requires more training time and computational resources.

2. The business may be willing to use this model for prediction if it produces clearly better results than the simpler models. However, management may be concerned because neural networks are difficult to interpret and do not clearly explain why a booking was classified as high risk. Therefore, the model would be more appropriate as a decision-support tool rather than as the only basis for operational decisions. Additional explanation methods and human review would improve trust in its predictions.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [11]:
#import pandas as pd

# Create a comparison table
model_comparison = pd.DataFrame({
    "Model": ["Naïve Bayes", "SVM", "Neural Network"],
    "Accuracy": [nb_accuracy, svm_accuracy, nn_accuracy],
    "Precision": [nb_precision, svm_precision, nn_precision],
    "Recall": [nb_recall, svm_recall, nn_recall],
    "F1-score": [nb_f1, svm_f1, nn_f1]
})

# Sort models by accuracy
model_comparison = model_comparison.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print("Model Comparison:")
print(model_comparison.to_string(index=False))

best_model = model_comparison.iloc[0]["Model"]
best_accuracy = model_comparison.iloc[0]["Accuracy"]

print(f"\nBest model by accuracy: {best_model}")
print(f"Best accuracy: {best_accuracy:.4f}")

Model Comparison:
         Model  Accuracy  Precision   Recall  F1-score
   Naïve Bayes  1.000000   1.000000 1.000000  1.000000
           SVM  1.000000   1.000000 1.000000  1.000000
Neural Network  0.999386   0.999397 0.998945  0.999171

Best model by accuracy: Naïve Bayes
Best accuracy: 1.0000


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The three models were compared using accuracy, precision, recall, and F1-score. Naïve Bayes was the fastest and easiest model to train and interpret, but it may make stronger independence assumptions and provide lower predictive performance. The linear SVM offered a balance between performance and complexity, although it required more training time. The neural network was the least interpretable and most computationally demanding, but it was able to capture more complex relationships in the booking data. The model with the highest accuracy in the comparison table had the best overall predictive performance.

2. I would recommend deploying the model with the highest accuracy and F1-score, as long as its improvement over the simpler models is meaningful. If the neural network performs only slightly better, I would recommend the SVM because it provides a better balance of accuracy, training time, ease of use, and interpretability. The selected model could be used to identify high-risk bookings, support inventory planning, improve staffing decisions, and guide reservation confirmation strategies.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend implementing the model with the highest F1-score and accuracy, provided that its improvement over simpler models is meaningful. The model can help predict booking cancellations, allowing management to improve staffing, room inventory, overbooking, and customer follow-up decisions. Important limitations include false predictions, limited interpretability, and possible changes in customer behavior over time. Future results could be improved by adding more recent booking data, testing additional features, tuning model parameters, and monitoring performance regularly after deployment.

2. This assignment relates to my customized learning outcome because it helped me apply machine learning models to a real business problem. I practiced preparing data, building classification models, evaluating performance with multiple metrics, comparing model strengths and limitations, and translating technical results into a practical business recommendation.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [12]:
!jupyter nbconvert --to html "assignment_9_bayes_svm_neural.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_9_bayes_svm_neural.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=T